In [4]:
import os
import re
import json

def parse_ann_file(file_path):
    entities = {}
    attributes = {}
    relations = []

    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            if line.startswith('T'):
                parts = line.strip().split('\t')
                if len(parts) == 3:
                    entity_id = parts[0]
                    entity_info = parts[1].split()
                    entity_type = entity_info[0]
                    entity_start = entity_info[1]
                    entity_end = entity_info[2]
                    entity_text = parts[2]
                    entities[entity_id] = {
                        'type': entity_type,
                        'start_offset': entity_start,
                        'end_offset': entity_end,
                        'text': entity_text
                    }
            elif line.startswith('A'):
                parts = line.strip().split('\t')
                if len(parts) == 3:
                    attr_id = parts[0]
                    attr_info = parts[1].split()
                    attr_type = attr_info[0]
                    attr_target = attr_info[1]
                    attr_value = attr_info[2]
                    attributes[attr_id] = {
                        'type': attr_type,
                        'target': attr_target,
                        'value': attr_value
                    }
            elif line.startswith('R'):
                parts = line.strip().split('\t')
                if len(parts) == 2:
                    relation_id = parts[0]
                    relation_info = parts[1].split()
                    relation_type = relation_info[0]
                    arg1 = relation_info[1].split(':')[1]
                    arg2 = relation_info[2].split(':')[1]
                    relations.append({
                        'type': relation_type,
                        'arg1': arg1,
                        'arg2': arg2
                    })

    return entities, attributes, relations

def convert_to_json(entities, attributes, relations):
    json_data = []

    for entity_id, entity in entities.items():
        entity_json = {
            'id': entity_id,
            'type': entity['type'],
            'start_offset': entity['start_offset'],
            'end_offset': entity['end_offset'],
            'text': entity['text']
        }
        json_data.append(entity_json)

    for attr_id, attr in attributes.items():
        attribute_json = {
            'id': attr_id,
            'type': attr['type'],
            'target': attr['target'],
            'value': attr['value']
        }
        json_data.append(attribute_json)

    for relation in relations:
        if relation['arg1'] in entities and relation['arg2'] in entities:
            relation_json = {
                'type': relation['type'],
                'arg1_text': entities[relation['arg1']]['text'],
                'arg1_offset': f"{entities[relation['arg1']]['start_offset']} {entities[relation['arg1']]['end_offset']}",
                'arg2_text': entities[relation['arg2']]['text'],
                'arg2_offset': f"{entities[relation['arg2']]['start_offset']} {entities[relation['arg2']]['end_offset']}"
            }
            json_data.append(relation_json)

    return json_data

def process_ann_files(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    for filename in os.listdir(input_folder):
        if filename.endswith(".ann"):
            file_path = os.path.join(input_folder, filename)
            entities, attributes, relations = parse_ann_file(file_path)
            json_data = convert_to_json(entities, attributes, relations)

            json_filename = f"{os.path.splitext(filename)[0]}.json"
            json_file_path = os.path.join(output_folder, json_filename)

            with open(json_file_path, 'w', encoding='utf-8') as json_file:
                json.dump(json_data, json_file, indent=4)

# Beispielaufruf
input_folder = "lct_ann"
output_folder = "all_entitys"
process_ann_files(input_folder, output_folder)

### Claude : 

In [3]:
import os
import re
import json

def parse_ann_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read()

    # Entitäten extrahieren
    entities = re.findall(r'(T\d+)\s+(\S+)\s+(\d+)\s+(\d+)\s+(.+)', content)
    entity_dict = {entity[0]: {'type': entity[1], 'start': int(entity[2]), 'end': int(entity[3]), 'text': entity[4]} for entity in entities}

    # Attribute extrahieren
    attributes = re.findall(r'(A\d+)\s+(\S+)\s+(T\d+)\s+(.+)', content)
    attribute_dict = {attr[0]: {'type': attr[1], 'target': attr[2], 'value': attr[3]} for attr in attributes}

    # Relationen extrahieren
    relations = re.findall(r'(R\d+)\s+(\S+)\s+Arg1:(T\d+)\s+Arg2:(T\d+)', content)
    relation_list = [{'type': rel[1], 'arg1': entity_dict[rel[2]], 'arg2': entity_dict[rel[3]]} for rel in relations]

    # Vergleiche extrahieren
    comparisons = re.findall(r'(E\d+)\s+(\S+):?(T\d+)?\s+(\S+):?(T\d+)?\s+(\S+):?(T\d+)?', content)
    comparison_list = []
    for comp in comparisons:
        comparison_dict = {'id': comp[0], 'type': comp[1]}
        for i in range(2, len(comp), 2):
            if comp[i] and comp[i+1]:
                comparison_dict[comp[i]] = entity_dict.get(comp[i+1], attribute_dict.get(comp[i+1]))
        comparison_list.append(comparison_dict)

    # Ergebnisse zusammenführen
    result = {
        'entities': list(entity_dict.values()),
        'attributes': list(attribute_dict.values()),
        'relations': relation_list,
        'comparisons': comparison_list
    }

    return result

def process_ann_files(input_directory, output_directory):
    # Erstelle den Ausgabeordner, falls er noch nicht existiert
    os.makedirs(output_directory, exist_ok=True)

    for filename in os.listdir(input_directory):
        if filename.endswith('.ann'):
            file_path = os.path.join(input_directory, filename)
            nct_number = re.findall(r'(NCT\d+)', filename)

            if nct_number:
                nct_number = nct_number[0]
                json_output = parse_ann_file(file_path)

                output_filename = f"{nct_number}_entities.json"
                output_path = os.path.join(output_directory, output_filename)

                with open(output_path, 'w', encoding='utf-8') as json_file:
                    json.dump(json_output, json_file, indent=2)

                print(f"Parsed {filename} and saved as {output_filename}")

# Beispielaufruf
input_directory = 'lct_ann'
output_directory = 'all_entities'
process_ann_files(input_directory, output_directory)

Parsed NCT03860012.ann and saved as NCT03860012_entities.json
Parsed NCT03860025.ann and saved as NCT03860025_entities.json
Parsed NCT03860038.ann and saved as NCT03860038_entities.json
Parsed NCT03860064.ann and saved as NCT03860064_entities.json
Parsed NCT03860090.ann and saved as NCT03860090_entities.json
Parsed NCT03860103.ann and saved as NCT03860103_entities.json
Parsed NCT03860116.ann and saved as NCT03860116_entities.json
Parsed NCT03860142.ann and saved as NCT03860142_entities.json
Parsed NCT03860168.ann and saved as NCT03860168_entities.json
Parsed NCT03860181.ann and saved as NCT03860181_entities.json
Parsed NCT03860194.ann and saved as NCT03860194_entities.json
Parsed NCT03860220.ann and saved as NCT03860220_entities.json
Parsed NCT03860233.ann and saved as NCT03860233_entities.json
Parsed NCT03860246.ann and saved as NCT03860246_entities.json
Parsed NCT03860259.ann and saved as NCT03860259_entities.json
Parsed NCT03860311.ann and saved as NCT03860311_entities.json
Parsed N